# 02 - Training and Evaluation

This notebook covers the training and evaluation phase of the Neural Machine Translation (NMT) project.
It loads the preprocessed data and vocabularies, initializes the sequence-to-sequence model (Encoder-Decoder with Attention), and runs the training loop using Teacher Forcing.

It also includes placeholders for optional tasks such as BLEU score evaluation and Beam Search decoding.

In [1]:
import os
import sys
import random
import math
import time

sys.path.append(os.path.abspath(os.path.join('..','src')))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Import our custom modules
from vocabulary import Vocabulary
from dataset import TranslationDataset, collate_fn, load_processed
from encoder import Encoder
from decoder import Decoder
from seq2seq import Seq2Seq

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

/home/julia-tomaszkiewicz/Dokumenty/Deep Learning/Machine_Translation/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


PyTorch version: 2.10.0+cu128
Using device: cpu


## 1. Configuration & Hyperparameters
Define the sizes of the neural network layers, learning rate, and batch size. This is where you can perform hyperparameter tuning.

In [2]:
data_dir = os.path.abspath(os.path.join('..','data','processed'))
models_dir = os.path.abspath(os.path.join('..','models'))

os.makedirs(models_dir, exist_ok=True)

# Hyperparameters
BATCH_SIZE = 32
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
ENC_HID_DIM = 512
DEC_HID_DIM = 512
ATTN_DIM = 256
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5
LEARNING_RATE = 0.001
N_EPOCHS = 10
CLIP = 1.0

# TODO (Optional): Implement the Transformer model (Self-Attention, Feed-Forward, Positional Encoding)
# Once implemented, we can add a flag here to switch between 'RNN' and 'TRANSFORMER'
MODEL_TYPE = 'RNN'

## 2. Loading Data
Load the `.pt` files and vocabularies created in the `01_data_preparation.ipynb` notebook.

In [3]:
# Load vocabularies
src_vocab = Vocabulary.load(os.path.join(data_dir, "vocab.src.json"))
tgt_vocab = Vocabulary.load(os.path.join(data_dir, "vocab.tgt.json"))

print(f"Source vocabulary size: {len(src_vocab)}")
print(f"Target vocabulary size: {len(tgt_vocab)}")

# Load datasets
train_data = load_processed(os.path.join(data_dir, "train.pt"))
val_data = load_processed(os.path.join(data_dir, "val.pt"))
test_data = load_processed(os.path.join(data_dir, "test.pt"))

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")
print(f"Testing examples: {len(test_data)}")

pad_idx = src_vocab.token2id[Vocabulary.PAD]

# Create DataLoaders
train_loader = DataLoader(TranslationDataset(train_data), batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: collate_fn(b, pad_id=pad_idx))
val_loader = DataLoader(TranslationDataset(val_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_fn(b, pad_id=pad_idx))
test_loader = DataLoader(TranslationDataset(test_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_fn(b, pad_id=pad_idx))

Source vocabulary size: 29961
Target vocabulary size: 48596
Training examples: 247865
Validation examples: 13770
Testing examples: 13771


## 3. Model Initialization
Instantiate the Encoder, Decoder, and the full Seq2Seq model. We also define the Optimizer (Adam) and the Loss Function (CrossEntropyLoss).

In [4]:
INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(tgt_vocab)
SOS_IDX = tgt_vocab.token2id[Vocabulary.SOS]
EOS_IDX = tgt_vocab.token2id[Vocabulary.EOS]

if MODEL_TYPE == 'RNN':
    enc = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, dropout=ENC_DROPOUT, pad_idx=pad_idx)
    dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, ENC_HID_DIM * 2, DEC_HID_DIM, ATTN_DIM, dropout=DEC_DROPOUT, pad_idx=pad_idx)
    model = Seq2Seq(enc, dec, device, pad_idx, SOS_IDX, EOS_IDX).to(device)
else:
    # TODO (Optional): Initialize Transformer model here. Added fake Encoder, Decoder and Seq2Seq classes for now to avoid errors.
    enc = Encoder()
    dec = Decoder()
    model = Seq2Seq().to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

The model has 113,283,028 trainable parameters


## 4. Training Loop
Define the functions for training (with Teacher Forcing) and evaluating (without Teacher Forcing) for a single epoch.

In [5]:
def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    
    for _, batch in enumerate(iterator):
        src, src_lens, tgt, _ = batch
        src, src_lens, tgt = src.to(device), src_lens.to(device), tgt.to(device)
        
        optimizer.zero_grad()
        output = model(src, src_lens, tgt)
        
        # Reshape for CrossEntropyLoss
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        tgt = tgt[:, 1:].reshape(-1)
        
        loss = criterion(output, tgt)
        loss.backward()
        
        # Clip gradients to prevent them from blowing up
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for _, batch in enumerate(iterator):
            src, src_lens, tgt, _ = batch
            src, src_lens, tgt = src.to(device), src_lens.to(device), tgt.to(device)
            
            # Turn off teacher forcing for evaluation (ratio = 0)
            output = model(src, src_lens, tgt, teacher_forcing_ratio=0)
            
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            tgt = tgt[:, 1:].reshape(-1)
            
            loss = criterion(output, tgt)
            epoch_loss += loss.item()
            
    return epoch_loss / len(iterator)

Execute training

In [6]:
best_valid_loss = float('inf')
model_filename = f"{MODEL_TYPE.lower()}_emb{ENC_EMB_DIM}_hid{ENC_HID_DIM}_lr{LEARNING_RATE}.pt"
model_save_path = os.path.join(models_dir, model_filename)
print(f"Starting training... Model will be saved to: {model_save_path}")

for epoch in range(N_EPOCHS):
    start_time = time.time()
    
    train_loss = train(model, train_loader, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, val_loader, criterion)
    
    end_time = time.time()
    
    # Save the model if it's the best one we've seen so far
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), model_save_path)
    
    # Perplexity (PPL) is a standard metric in NLP, calculated as exp(loss)
    print(f'Epoch: {epoch+1:02} | Time: {int(end_time - start_time)}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

Starting training... Model will be saved to: /home/julia-tomaszkiewicz/Dokumenty/Deep Learning/Machine_Translation/models/rnn_emb256_hid512_lr0.001.pt


KeyboardInterrupt: 

## 5. Inference and Evaluation
Load the best model and translate sentences from the test set.

In [ ]:
# Load best weights
model.load_state_dict(torch.load(model_save_path))

def strip_special(tokens):
    return [t for t in tokens if t not in (Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK)]

# Translate a few sentences from the test set
print("\n--- Example Translations ---")
model.eval()
with torch.no_grad():
    batch = next(iter(test_loader))
    src, src_lens, tgt, _ = batch
    src, src_lens, tgt = src.to(device), src_lens.to(device), tgt.to(device)
    
    # Use the greedy translation method implemented in seq2seq.py
    # TODO (Optional): Implement Beam Search in seq2seq.py and use it here instead of greedy decoding
    translations, attentions = model.translate(src, src_lens, max_len=50)
    
    for i in range(min(5, src.size(0))): # Show up to 5 examples
        src_sentence = ' '.join(strip_special(src_vocab.decode(src[i].tolist())))
        tgt_sentence = ' '.join(strip_special(tgt_vocab.decode(tgt[i].tolist())))
        pred_sentence = ' '.join(strip_special(tgt_vocab.decode(translations[i].tolist())))
        
        print(f"\nSource: {src_sentence}")
        print(f"Target: {tgt_sentence}")
        print(f"Predicted: {pred_sentence}")

In [ ]:
import nltk

# TODO: Implement BLEU score calculation over the entire test set.
# You can use `nltk.translate.bleu_score.corpus_bleu` for this.
# 1. Loop through test_loader.
# 2. Get predictions using model.translate().
# 3. Decode predictions and targets into lists of words.
# 4. Pass them to the BLEU score function.

print("\nTODO: Calculate final BLEU score on test_data.")